# EP08｜輝達 GPU 為什麼是 AI 主力？CPU 和 GPU 差在哪

像素腦底加！這是影片裡的實測：同樣的工作，「一個一個算」和「一次交一大批」差多少。按上方的 **執行階段 → 全部執行**（或每格按 ▶️）就能自己跑一遍。

⚠️ 第 1 到 4 節的兩種寫法都在同一顆 CPU 上跑，用來看懂「一次算一大批」的想法；每台電腦的秒數都不一樣，你的數字不會和影片完全相同。第 5 節才是真的 CPU 對 GPU（要先把執行階段換成 GPU）。

📺 影片：[youtube.com/@pixelbraintw](https://www.youtube.com/@pixelbraintw)

## 1. 把貓咪照片調亮
每個數字乘 1.5，超過 255 就停在 255。照片有 800 × 800 個像素，每個像素紅綠藍 3 個數字，總共 192 萬個數字。

In [ ]:
import time
from urllib.request import urlretrieve
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

urlretrieve("https://raw.githubusercontent.com/pixelbraintw/pixelbrain-code/main/ep02-edges/cat.png", "cat.png")
cat = np.asarray(Image.open("cat.png").convert("RGB"))
flat = cat.reshape(-1).tolist()
print(cat.shape, "共", cat.size, "個數字")

def best(f, n=3):
    """跑 n 次，取最快的一次（秒）。"""
    ts = []
    for _ in range(n):
        t = time.perf_counter(); f(); ts.append(time.perf_counter() - t)
    return min(ts)

In [ ]:
def one_by_one():
    out = []
    for v in flat:                      # 一個一個算
        out.append(min(255, int(v * 1.5)))
    return out

def all_at_once():
    return np.minimum(255, (cat * 1.5).astype(np.int32)).astype(np.uint8)   # 一次交一大批

a = np.array(one_by_one(), dtype=np.uint8).reshape(cat.shape)
b = all_at_once()
print("兩種寫法的結果一樣嗎？", (a == b).all())
t1, t2 = best(one_by_one), best(all_at_once, 10)
print(f"一個一個算：{t1:.4f} 秒")
print(f"一次交一大批：{t2:.4f} 秒，快了 {t1 / t2:.0f} 倍")
fig, ax = plt.subplots(1, 2, figsize=(9, 4.5))
ax[0].imshow(cat); ax[0].set_title("before"); ax[1].imshow(b); ax[1].set_title("after"); [x.axis("off") for x in ax]; plt.show()

## 2. 矩陣乘法：AI 裡最大量的計算
一排數字和另一排數字，一個對一個相乘，再全部加起來。`n = 400` 那一次要等幾秒到幾十秒。

In [ ]:
def matmul_loop(A, B):
    n, m, p = len(A), len(B), len(B[0])
    C = [[0.0] * p for _ in range(n)]
    for i in range(n):
        for j in range(p):
            s = 0.0
            for k in range(m):
                s += A[i][k] * B[k][j]
            C[i][j] = s
    return C

rng = np.random.default_rng(0)
for n in (50, 100, 200, 400):
    A, B = rng.random((n, n)), rng.random((n, n))
    Al, Bl = A.tolist(), B.tolist()
    assert np.allclose(matmul_loop(Al, Bl), A @ B)          # 兩種算法答案一樣
    t1 = best(lambda: matmul_loop(Al, Bl), 1 if n >= 200 else 3)
    t2 = best(lambda: A @ B, 20)
    print(f"{n} × {n}（{n**3:,} 次乘法和加法）：一個一個算 {t1:.4f} 秒，一次交一大批 {t2:.6f} 秒，快了 {t1 / t2:.0f} 倍")

## 3. 故意做錯：一步接一步的工作
每一步都要等上一步的答案，沒辦法分給很多人同時做。硬要用 NumPy，一次還是只能交一個數字。

In [ ]:
N = 1_000_000
def chain_loop():
    x = 0.0
    for _ in range(N):
        x = x * 0.999 + 1.0
    return x

one = np.float64(0.999); inc = np.float64(1.0)
def chain_numpy():
    x = np.float64(0.0)
    for _ in range(N):
        x = x * one + inc          # 硬要用 NumPy，一次還是只能算一個
    return x

t1, t2 = best(chain_loop), best(chain_numpy)
print(f"Python 一個一個算：{t1:.4f} 秒；硬要用 NumPy：{t2:.4f} 秒，慢了 {t2 / t1:.1f} 倍")

## 4. 故意做錯：工作太少
只有 10 個數字，交接比做事還久。把 `range(10)` 改成 `range(1000)` 再跑一次，看看誰贏。

In [ ]:
small = list(range(10))
def small_loop(): return [v + 1 for v in small]
def small_np(): return (np.array(small) + 1)

reps = 100_000
t1 = best(lambda: [small_loop() for _ in range(reps)]) / reps
t2 = best(lambda: [small_np() for _ in range(reps)]) / reps
print(f"Python：{t1 * 1e6:.2f} 微秒；NumPy：{t2 * 1e6:.2f} 微秒，慢了 {t2 / t1:.1f} 倍")

## 5. 真的 CPU 對 GPU（選做）
先到 **執行階段 → 變更執行階段類型**，硬體加速器選 **GPU**（例如 T4），再全部重新執行。
沒有 GPU 也能跑，只是只會印出 CPU 的時間。試試把 `n` 改成 100：工作太小的時候，GPU 還會贏嗎？

In [ ]:
import torch

def bench(n, device, reps=5):
    A = torch.rand(n, n, device=device); B = torch.rand(n, n, device=device)
    A @ B                                          # 先暖身一次
    if device == "cuda": torch.cuda.synchronize()
    t = time.perf_counter()
    for _ in range(reps):
        A @ B
    if device == "cuda": torch.cuda.synchronize()  # 等 GPU 真的算完才停錶
    return (time.perf_counter() - t) / reps

for n in (100, 1000, 4000):
    t_cpu = bench(n, "cpu")
    if torch.cuda.is_available():
        t_gpu = bench(n, "cuda")
        print(f"{n} × {n}：CPU {t_cpu:.5f} 秒，GPU（{torch.cuda.get_device_name(0)}）{t_gpu:.5f} 秒，GPU 快 {t_cpu / t_gpu:.1f} 倍")
    else:
        print(f"{n} × {n}：CPU {t_cpu:.5f} 秒（這個執行階段沒有 GPU）")

---
貓咪照片：AdinaVoicu，CC0，[Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Tabby_cat_with_blue_eyes-3336579.jpg)

覺得有趣的話，記得到 YouTube 訂閱 **像素腦**！有問題歡迎在影片留言 🐱